> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 05 · Deploy CampusMate to AgentCore Runtime

**CampusMate v2 (your agent, live in the cloud)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

Your CampusMate from notebook 04 runs only while your notebook is open, and only for you. To serve hundreds of students at 2 AM it needs a home in the cloud. **Amazon Bedrock AgentCore Runtime** is that home: you hand over your code, it gives you a secure endpoint, scales it, and keeps every student's conversation separate.

### What you'll do
- See the prototype-to-production gap
- Create an AgentCore project with the AgentCore CLI
- Put **your notebook-04 agent** inside it and add the 3-line AgentCore wrapper
- Deploy it with one command, then talk to it
- Prove sessions: same session remembers, a new one starts fresh

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else.

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## The prototype-to-production gap

| On your laptop (notebook 04) | In production |
|---|---|
| One user: you | Hundreds of students at once |
| Runs while the notebook is open | Always available, pay only when used |
| Uses your AWS keys | Gets its own AWS identity (a *role*) with just what it needs |
| `print()` | Logs, traces and metrics (Lab 9) |
| One conversation | Thousands of separate conversations |

🏫 **Analogy:** cooking for yourself vs running the college canteen at lunch hour.

## What AgentCore Runtime gives you

- **A separate little machine per conversation**: each *session* runs in its own isolated micro virtual machine (microVM).
- **A simple contract**: your code answers `POST /invocations`. The `BedrockAgentCoreApp` class does that for you; you only mark one
  function with `@app.entrypoint`.
- **Sessions**: every request carries a session ID (at least 33 characters). Same ID → same microVM → it still has the conversation.

We use the official **AgentCore CLI** (`agentcore`). It creates the project, the AWS role, the package and the Runtime for you.

## Step 1: create your AgentCore project

This makes a folder called `campusmate<your id>` next to `notebooks/`, with a starter agent inside. It takes about a minute.

(In a terminal this is `agentcore create --name campusmate<your id> --template agent-python-minimal`.)

In [ ]:
if PROJECT_DIR.exists():
    print("Your project already exists:", PROJECT_DIR)
else:
    agentcore("create", "--name", PROJECT, "--template", "agent-python-minimal", "--skip-git", folder=ROOT)

## Step 2: look at what the CLI made

The important files:
- `app/agent/main.py`: **your agent's code** (we replace it in Step 3)
- `app/agent/pyproject.toml`: the Python libraries your agent needs
- `agentcore/agentcore.json`: what your project contains (right now: one Runtime called `agent`)

Here is the starter `main.py`. Notice the 3 AgentCore lines: create the `app`, mark a function with `@app.entrypoint`, and `app.run()`.

In [ ]:
print(AGENT_FILE.read_text())

## Step 3: put YOUR CampusMate inside

This is the agent you built in notebook 04: the same `get_course_info` and `get_campus_policy` tools, the same idea.
The only new part is at the bottom: the AgentCore wrapper.

- `invoke(payload, context)` runs for every message. `payload` is what the student sent; `context.session_id` says which conversation it belongs to.
- We keep one agent per session in a small dictionary, so each conversation remembers its own messages.

In [ ]:
%%writefile {AGENT_FILE}
import json
from pathlib import Path
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

DATA_DIR = Path(__file__).parent / "data"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # Amazon Nova 2 Lite


# ---- Your tools from notebook 04 (unchanged) ----
@tool
def get_course_info(query: str) -> str:
    """Look up AIDD courses by course code (e.g. AIDD303) or a keyword in the title/description (e.g. docker).

    Args:
        query: A course code or a keyword.
    """
    q = query.lower()
    courses = json.loads((DATA_DIR / "courses.json").read_text())["courses"]
    hits = [c for c in courses if q == c["code"].lower() or q in (c["title"] + " " + c["description"]).lower()]
    return json.dumps(hits[:5]) if hits else json.dumps({"found": False, "codes": [c["code"] for c in courses]})


@tool
def get_campus_policy(topic: str) -> str:
    """Return CampusMate rules for a topic: attendance, internal_assessments, semester_end_exams, fees, leave,
    re_evaluation, library, hackathons. Always use this instead of guessing a rule.

    Args:
        topic: one of the topics above
    """
    policies = json.loads((DATA_DIR / "campus_policies.json").read_text())["policies"]
    return json.dumps(policies.get(topic.lower().strip(), {"found": False, "topics": list(policies)}))


# ---- NEW in Lab 5: wrap the agent for AgentCore Runtime ----
app = BedrockAgentCoreApp()
agents = {}   # one agent per conversation (session)


@app.entrypoint
def invoke(payload, context):
    student = payload.get("student_id", "AIDD-DEMO-001")
    session = context.session_id
    if session not in agents:
        agents[session] = Agent(
            model=BedrockModel(model_id=MODEL_ID, max_tokens=1200),
            tools=[get_course_info, get_campus_policy],
            system_prompt=f"You are CampusMate for AIDD students, talking to student {student}. "
                          "Use tools for courses and rules; never invent rules. Be brief.",
        )
    answer = agents[session](payload["prompt"])
    return {"result": str(answer)}


if __name__ == "__main__":
    app.run()

Your agent reads the fictional campus files, so we copy `data/` into the project. Then we add the libraries it needs (Strands, plus one we'll use in Lab 7).

In [ ]:
shutil.copytree(DATA_DIR, AGENT_FILE.parent / "data", dirs_exist_ok=True)

pyproject = AGENT_FILE.parent / "pyproject.toml"          # one of the libraries needs Python 3.11 or newer
pyproject.write_text(pyproject.read_text().replace('requires-python = ">=3.10"', 'requires-python = ">=3.11"'))
subprocess.run([shutil.which("uv") or "uv", "add", "--quiet", "strands-agents==1.64.0", "mcp-proxy-for-aws==1.8.0"],
               cwd=AGENT_FILE.parent, check=True)
print("Libraries added. Your agent folder now has:", sorted(p.name for p in AGENT_FILE.parent.iterdir()))

## Step 4: deploy

One command. Behind the scenes `agentcore deploy`:
1. creates an AWS **role** (an ID card) for your agent, allowed to call Bedrock models and write logs,
2. packages your code and its libraries for the cloud's computers,
3. creates the **AgentCore Runtime** and switches on tracing (Lab 9).

The first deploy takes **2-3 minutes**. Read the messages as they appear.

In [ ]:
agentcore("deploy")

## What you built in earlier labs

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## Step 5: talk to your deployed agent

`ask()` (defined above) uses `agentcore invoke`. The terminal version is:

`agentcore invoke --runtime agent --session-id <id> --payload '{"prompt": "..."}'`

In [ ]:
session_a = f"lab05-{uuid.uuid4()}"      # a session ID: 42 characters (Runtime needs at least 33)
print(ask("Hi, I'm Kiran. How is the Docker course assessed?", session_a))

## Step 6: prove sessions work

Same session again, then a brand-new session.

In [ ]:
print("Same session :", ask("What's my name?", session_a))
session_b = f"lab05-{uuid.uuid4()}"
print("New session  :", ask("What's my name?", session_b))

Session A remembered *Kiran*, session B did not. That memory lives **inside session A's microVM** and disappears when the session
ends (after about 15 idle minutes). Memory that lasts across sessions is the next lab.

## ✍️ Your turn

Ask your agent two questions in a **new** session: one about a course and one about a campus rule. Check it uses its tools.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
MY_SESSION = "TODO"   # TODO: make a new session ID, e.g. f"mine-{uuid.uuid4()}"

if MY_SESSION != "TODO":
    print(ask("TODO: a question about a course", MY_SESSION))
    print(ask("TODO: a question about a campus rule", MY_SESSION))
else:
    print("Skipped: set MY_SESSION first.")

## ✅ Checkpoint

You are on track if:
- Step 1 created the folder `campusmate<your id>`
- Step 4 ended with `deployed project ...`
- Session A remembered Kiran; session B did not

**Stuck or behind?** Copy from `the Step 3 cell (it is the whole agent file). The finished agent is also in projects/campusmate_agent.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `'agentcore' is not recognized` / `FileNotFoundError: agentcore` | The AgentCore CLI isn't installed | Run `npm install -g @aws/agentcore` (needs Node.js 20+), then restart Jupyter |
| `uv: command not found` | uv isn't installed | Install uv (SETUP.md), then restart Jupyter |
| Deploy fails with a CDK / bootstrap message | The account needs a one-time CDK setup | Ask your instructor (they run `cdk bootstrap` once per account) |
| `ValidationException ... runtimeSessionId` | Session ID shorter than 33 characters | Use `f"lab05-{uuid.uuid4()}"` |
| Project name error | Project names allow only letters and digits | Use a STUDENT_ID with lower-case letters and digits only |